# LeetCode 552 — Student Attendance Record II bằng Python

- Link: https://leetcode.com/problems/student-attendance-record-ii/
- Độ khó: **Hard**
- Chủ đề: Quy hoạch động (DP) trạng thái, Đếm tổ hợp, Lũy thừa ma trận, Modulo

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def checkRecord(self, n: int) -> int:

```

## 1. Giới thiệu & mục tiêu học

Bài này là ví dụ đẹp nhất để học **DP theo trạng thái (state machine DP)**: ta xây chuỗi từng ký tự một, và chỉ cần nhớ **một vài con số tóm tắt** về phần đã xây (đã vắng bao nhiêu lần, đang đi muộn liên tiếp mấy ngày) thay vì nhớ cả chuỗi.

Sau notebook này bạn sẽ:

1. Biết cách biến một điều kiện "chuỗi hợp lệ" thành **máy trạng thái** với số trạng thái nhỏ.
2. Viết DP bảng `dp[i][a][l]`, rồi rút gọn còn **6 biến** chạy `O(n)` thời gian, `O(1)` bộ nhớ.
3. Hiểu cách **tách bài toán**: đếm chuỗi không có `A` trước, rồi "chèn" `A` vào — một mẹo tổ hợp rất hay.
4. Dùng **lũy thừa ma trận** để đếm trong `O(log n)` — giải được cả khi `n = 10^18`.
5. Làm việc an toàn với **modulo `10^9 + 7`** và kiểm chứng bằng brute force.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Hồ sơ điểm danh của một học sinh là chuỗi gồm các ký tự:

- `'A'`: Absent — vắng mặt,
- `'L'`: Late — đi muộn,
- `'P'`: Present — có mặt.

Học sinh **đủ điều kiện nhận thưởng** nếu thỏa **cả hai**:

1. Tổng số lần vắng `'A'` **ít hơn 2** (tức là 0 hoặc 1 lần).
2. **Không bao giờ** đi muộn 3 ngày liên tiếp trở lên (chuỗi không chứa `"LLL"`).

Cho `n`, đếm số hồ sơ độ dài `n` đủ điều kiện. Kết quả có thể rất lớn nên trả về **modulo `10^9 + 7`**.

**Ví dụ 1**

```text
Input:  n = 2
Output: 8
Giải thích: có 3^2 = 9 chuỗi độ dài 2; chỉ "AA" bị loại (vắng 2 lần).
"PP", "AP", "PA", "LP", "PL", "AL", "LA", "LL"
```

**Ví dụ 2**

```text
Input:  n = 1
Output: 3
```

**Ví dụ 3**

```text
Input:  n = 10101
Output: 183236316
```

**Ràng buộc**

- `1 <= n <= 10^5`

Lưu ý: không thể liệt kê `3^n` chuỗi khi `n = 10^5` — phải **đếm thông minh**.

In [1]:
from itertools import product

def is_rewardable(s):
    return s.count('A') < 2 and 'LLL' not in s

good = [''.join(t) for t in product('PAL', repeat=2) if is_rewardable(''.join(t))]
print(len(good), good)
print("AA hợp lệ?", is_rewardable("AA"), "| PLLLP hợp lệ?", is_rewardable("PLLLP"), "| LLPLL hợp lệ?", is_rewardable("LLPLL"))

8 ['PP', 'PA', 'PL', 'AP', 'AL', 'LP', 'LA', 'LL']
AA hợp lệ? False | PLLLP hợp lệ? False | LLPLL hợp lệ? True


## 3. Kiến thức nền

### 3.1. `itertools.product` — sinh mọi chuỗi độ dài `n`

`product('PAL', repeat=n)` sinh mọi bộ `n` ký tự, mỗi ký tự lấy từ `'PAL'` → đúng `3^n` bộ. Dùng cho brute force với `n` nhỏ.

In [2]:
from itertools import product
print(list(product('AB', repeat=3)))
print("số chuỗi PAL độ dài 4:", sum(1 for _ in product('PAL', repeat=4)))

[('A', 'A', 'A'), ('A', 'A', 'B'), ('A', 'B', 'A'), ('A', 'B', 'B'), ('B', 'A', 'A'), ('B', 'A', 'B'), ('B', 'B', 'A'), ('B', 'B', 'B')]
số chuỗi PAL độ dài 4: 81


### 3.2. Modulo và tính chất cộng / nhân

Với `MOD = 10^9 + 7`: `(a + b) % MOD = ((a % MOD) + (b % MOD)) % MOD`, tương tự cho phép nhân. Vì vậy ta có thể lấy `% MOD` **sau mỗi bước** để số luôn nhỏ — Python không tràn số nhưng số khổng lồ làm chậm rất nhiều.

In [3]:
MOD = 10**9 + 7
a, b = 10**18 + 5, 10**18 + 9
print((a + b) % MOD == ((a % MOD) + (b % MOD)) % MOD)
print((a * b) % MOD == ((a % MOD) * (b % MOD)) % MOD)
# Phép trừ: kết quả % trong Python luôn không âm (khác C++/Java)
print((3 - 5) % MOD)

True
True
1000000005


### 3.3. Gán đồng thời nhiều biến (tuple unpacking)

Khi cập nhật DP "từ trạng thái cũ sang mới", ta cần **mọi vế phải dùng giá trị cũ**. Gán đồng thời `x, y = y, x + y` làm đúng điều đó: Python tính hết vế phải trước rồi mới gán.

In [4]:
x, y = 1, 1
for _ in range(5):
    x, y = y, x + y        # đúng: dùng x, y cũ
print("Fibonacci:", x, y)

x, y = 1, 1
for _ in range(5):
    x = y                  # sai: x đã bị ghi đè trước khi tính y
    y = x + y
print("Sai thứ tự:", x, y)

Fibonacci: 8 13
Sai thứ tự: 16 32


### 3.4. Nhân ma trận và lũy thừa nhanh

Lũy thừa nhanh (binary exponentiation): `a^13 = a^8 · a^4 · a^1` (vì `13 = 1101₂`). Chỉ cần `O(log n)` phép nhân. Áp dụng y hệt cho **ma trận vuông**.

In [5]:
def mat_mul(A, B, mod):
    n, m, p = len(A), len(B), len(B[0])
    C = [[0] * p for _ in range(n)]
    for i in range(n):
        for k in range(m):
            if A[i][k]:
                aik = A[i][k]
                for j in range(p):
                    C[i][j] = (C[i][j] + aik * B[k][j]) % mod
    return C

def mat_pow(M, e, mod):
    n = len(M)
    R = [[int(i == j) for j in range(n)] for i in range(n)]   # ma trận đơn vị
    while e:
        if e & 1:
            R = mat_mul(R, M, mod)
        M = mat_mul(M, M, mod)
        e >>= 1
    return R

# Fibonacci bằng ma trận [[1,1],[1,0]]^n
F = mat_pow([[1, 1], [1, 0]], 10, 10**9 + 7)
print("F(10) =", F[0][1])
print("13 nhị phân:", bin(13))

F(10) = 55
13 nhị phân: 0b1101


### 3.5. Dãy "tribonacci"

Dãy `t[i] = t[i-1] + t[i-2] + t[i-3]` xuất hiện khi đếm chuỗi **không có 3 ký tự giống nhau liên tiếp** của một loại. Ta sẽ gặp lại ở Cách 4.

In [6]:
t = [1, 2, 4]
for i in range(3, 10):
    t.append(t[-1] + t[-2] + t[-3])
print(t)

[1, 2, 4, 7, 13, 24, 44, 81, 149, 274]


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Ta cần nhớ gì khi xây chuỗi từ trái sang phải?

Giả sử đã viết xong `i` ký tự đầu và chuỗi đến giờ vẫn hợp lệ. Muốn biết thêm ký tự tiếp theo có được không, ta chỉ cần biết:

- `a` = số lần `A` đã dùng: **0 hoặc 1** (nếu là 2 thì đã hỏng).
- `l` = số `L` **liên tiếp ở cuối** chuỗi: **0, 1 hoặc 2** (nếu là 3 thì đã hỏng).

→ Chỉ có `2 × 3 = 6` trạng thái! Không cần nhớ cả chuỗi.

### 4.2. Bảng chuyển trạng thái

| Thêm ký tự | Từ `(a, l)` sang | Điều kiện |
|---|---|---|
| `P` | `(a, 0)` | luôn được |
| `A` | `(a + 1, 0)` | chỉ khi `a = 0` |
| `L` | `(a, l + 1)` | chỉ khi `l < 2` |

### 4.3. Tính tay `n = 1, 2`

Bắt đầu: chuỗi rỗng ở trạng thái `(0, 0)` với 1 cách.

**Sau 1 ký tự:**
- `P` → `(0,0)`: 1
- `A` → `(1,0)`: 1
- `L` → `(0,1)`: 1
- Tổng 3 ✓.

**Sau 2 ký tự** (đếm vào từng trạng thái):
- `(0,0)`: thêm `P` vào `(0,0)` hoặc `(0,1)` → 1 + 1 = 2 (`PP`, `LP`)
- `(0,1)`: thêm `L` vào `(0,0)` → 1 (`PL`)
- `(0,2)`: thêm `L` vào `(0,1)` → 1 (`LL`)
- `(1,0)`: thêm `A` vào `(0,0)`, `(0,1)` → 2 (`PA`, `LA`); thêm `P` vào `(1,0)` → 1 (`AP`). Tổng 3
- `(1,1)`: thêm `L` vào `(1,0)` → 1 (`AL`)
- `(1,2)`: 0
- Tổng: 2 + 1 + 1 + 3 + 1 + 0 = **8** ✓.

In [7]:
# Mô phỏng đúng quá trình tính tay ở trên, in số cách theo từng trạng thái
from collections import Counter

def step(cnt):
    new = Counter()
    for (a, l), c in cnt.items():
        new[(a, 0)] += c                  # P
        if a == 0:
            new[(1, 0)] += c              # A
        if l < 2:
            new[(a, l + 1)] += c          # L
    return new

cnt = Counter({(0, 0): 1})
for i in range(1, 4):
    cnt = step(cnt)
    print(f"sau {i} ký tự:", dict(sorted(cnt.items())), "-> tổng", sum(cnt.values()))

sau 1 ký tự: {(0, 0): 1, (0, 1): 1, (1, 0): 1} -> tổng 3
sau 2 ký tự: {(0, 0): 2, (0, 1): 1, (0, 2): 1, (1, 0): 3, (1, 1): 1} -> tổng 8
sau 3 ký tự: {(0, 0): 4, (0, 1): 2, (0, 2): 1, (1, 0): 8, (1, 1): 3, (1, 2): 1} -> tổng 19


## 5. Cách 1 — Brute Force: sinh mọi chuỗi

**Ý tưởng.** Sinh cả `3^n` chuỗi bằng `product`, đếm chuỗi thỏa `is_rewardable`. Chắc chắn đúng — dùng để kiểm chứng.

In [8]:
def check_record_brute(n):
    return sum(1 for t in product('PAL', repeat=n) if is_rewardable(''.join(t)))

for n in range(1, 8):
    print(n, check_record_brute(n))
assert check_record_brute(1) == 3
assert check_record_brute(2) == 8

1 3
2 8
3 19
4 43
5 94
6 200
7 418


**Độ phức tạp:** Time `O(3^n · n)`, Space `O(n)`. `n = 15` đã là 14 triệu chuỗi; `n = 10^5` là không tưởng.

**Nút thắt:** ta phân biệt từng chuỗi, trong khi nhiều chuỗi có **cùng "tương lai"** (cùng `a`, cùng số `L` ở cuối). Gom chúng lại theo trạng thái là bước đột phá.

In [9]:
import time
for n in (8, 10, 12):
    t = time.perf_counter()
    check_record_brute(n)
    print(f"n={n}: {time.perf_counter() - t:.3f}s")

n=8: 0.001s
n=10: 0.008s
n=12: 0.074s


## 6. Bước đột phá

**Insight:** Hai chuỗi tiền tố có cùng `(a, l)` thì **mọi cách viết tiếp** hợp lệ cho chuỗi này cũng hợp lệ cho chuỗi kia (điều kiện tương lai chỉ phụ thuộc `a` và `l`). Vậy ta chỉ cần **đếm số tiền tố theo trạng thái**, không cần biết từng chuỗi.

**Invariant:** sau bước `i`, `dp[a][l]` = số chuỗi độ dài `i` hợp lệ, có đúng `a` chữ `A`, kết thúc bằng đúng `l` chữ `L` liên tiếp.

**Quy tắc:**
- Nếu thêm `P` thì `l` về 0, `a` giữ nguyên.
- Nếu thêm `A` (chỉ khi `a = 0`) thì `a = 1`, `l` về 0.
- Nếu thêm `L` (chỉ khi `l < 2`) thì `l` tăng 1.

Đáp án = tổng 6 giá trị `dp[a][l]` sau bước `n`.

Viết thành công thức cho từng trạng thái mới (vế phải là giá trị cũ):

```text
new[0][0] = dp[0][0] + dp[0][1] + dp[0][2]                       (thêm P)
new[0][1] = dp[0][0]                                             (thêm L)
new[0][2] = dp[0][1]                                             (thêm L)
new[1][0] = (dp[0][0] + dp[0][1] + dp[0][2])                     (thêm A)
          + (dp[1][0] + dp[1][1] + dp[1][2])                     (thêm P)
new[1][1] = dp[1][0]
new[1][2] = dp[1][1]
```

In [10]:
# Kiểm chứng insight: các chuỗi độ dài 3 gom theo trạng thái, đếm số cách viết tiếp 2 ký tự hợp lệ
def state_of(s):
    a = s.count('A')
    l = len(s) - len(s.rstrip('L'))
    return (a, l)

futures = {}
for t in product('PAL', repeat=3):
    s = ''.join(t)
    if not is_rewardable(s):
        continue
    ext = sum(1 for u in product('PAL', repeat=2) if is_rewardable(s + ''.join(u)))
    futures.setdefault(state_of(s), set()).add(ext)
for st, e in sorted(futures.items()):
    print(f"trạng thái {st}: số cách viết tiếp = {e}")   # mỗi trạng thái chỉ có MỘT giá trị

trạng thái (0, 0): số cách viết tiếp = {8}
trạng thái (0, 1): số cách viết tiếp = {7}
trạng thái (0, 2): số cách viết tiếp = {5}
trạng thái (1, 0): số cách viết tiếp = {4}
trạng thái (1, 1): số cách viết tiếp = {3}
trạng thái (1, 2): số cách viết tiếp = {2}


## 7. Các cách tiếp theo

Notebook trình bày **5 cách**:

| # | Cách | Ý chính |
|---|---|---|
| 1 | Brute force (mục 5) | sinh `3^n` chuỗi |
| 2 | DP bảng `dp[i][a][l]` | máy trạng thái 6 trạng thái, lưu đủ `n` tầng |
| 3 | DP 6 biến | như Cách 2 nhưng chỉ giữ tầng hiện tại |
| 4 | Tách chữ `A` | đếm chuỗi chỉ có `P/L`, rồi chọn vị trí đặt `A` |
| 5 | Lũy thừa ma trận | `O(log n)` |

### 7.1. Cách 2 — DP bảng `dp[i][a][l]`

**Giải thích.** Tạo mảng 3 chiều kích thước `(n+1) × 2 × 3`. `dp[0][0][0] = 1` (chuỗi rỗng). Với mỗi `i` và mỗi trạng thái, "đẩy" số cách sang trạng thái mới theo 3 ký tự. Cách viết "đẩy" (push) dễ hiểu cho người mới vì nó đi đúng theo bảng chuyển.

In [11]:
MOD = 10**9 + 7

def check_record_table(n):
    dp = [[[0] * 3 for _ in range(2)] for _ in range(n + 1)]
    dp[0][0][0] = 1
    for i in range(n):
        for a in range(2):
            for l in range(3):
                c = dp[i][a][l]
                if c == 0:
                    continue
                # thêm 'P'
                dp[i + 1][a][0] = (dp[i + 1][a][0] + c) % MOD
                # thêm 'A'
                if a == 0:
                    dp[i + 1][1][0] = (dp[i + 1][1][0] + c) % MOD
                # thêm 'L'
                if l < 2:
                    dp[i + 1][a][l + 1] = (dp[i + 1][a][l + 1] + c) % MOD
    return sum(dp[n][a][l] for a in range(2) for l in range(3)) % MOD

print([check_record_table(n) for n in range(1, 8)])
assert check_record_table(2) == 8
assert check_record_table(10101) == 183236316

[3, 8, 19, 43, 94, 200, 418]


**Đọc code:**

- `dp[i][a][l]`: số chuỗi độ dài `i` ở trạng thái `(a, l)`.
- `if c == 0: continue` — bỏ qua trạng thái chưa có chuỗi nào (ví dụ `(1, 2)` ở `i = 0`), tiết kiệm thời gian.
- Ba khối "thêm ký tự" đúng bằng bảng chuyển ở mục 4.2.
- Lấy `% MOD` ngay khi cộng.

**Code nộp LeetCode (Cách 2):**

In [12]:
class SolutionTable:
    def checkRecord(self, n: int) -> int:
        MOD = 10**9 + 7
        dp = [[[0] * 3 for _ in range(2)] for _ in range(n + 1)]
        dp[0][0][0] = 1
        for i in range(n):
            for a in range(2):
                for l in range(3):
                    c = dp[i][a][l]
                    if not c:
                        continue
                    dp[i + 1][a][0] = (dp[i + 1][a][0] + c) % MOD
                    if a == 0:
                        dp[i + 1][1][0] = (dp[i + 1][1][0] + c) % MOD
                    if l < 2:
                        dp[i + 1][a][l + 1] = (dp[i + 1][a][l + 1] + c) % MOD
        return sum(map(sum, dp[n])) % MOD

assert SolutionTable().checkRecord(1) == 3
assert SolutionTable().checkRecord(2) == 8
print("SolutionTable OK")

SolutionTable OK


**Độ phức tạp Cách 2:** Time `O(6n) = O(n)`, Space `O(6n) = O(n)`. Với `n = 10^5`: 600 nghìn ô — chạy được nhưng tốn bộ nhớ không cần thiết, vì tầng `i+1` chỉ cần tầng `i`.

### 7.2. Cách 3 — DP 6 biến (rolling)

**Giải thích.** Đặt tên 6 trạng thái cho dễ đọc:

| Biến | Trạng thái | Ý nghĩa |
|---|---|---|
| `p0` | `(0,0)` | chưa vắng, cuối không phải L |
| `l01` | `(0,1)` | chưa vắng, cuối có đúng 1 L |
| `l02` | `(0,2)` | chưa vắng, cuối có đúng 2 L |
| `p1` | `(1,0)` | đã vắng 1 lần, cuối không phải L |
| `l11` | `(1,1)` | đã vắng 1 lần, cuối 1 L |
| `l12` | `(1,2)` | đã vắng 1 lần, cuối 2 L |

Dùng công thức "kéo" (pull) ở mục 6, gán đồng thời để mọi vế phải là giá trị cũ.

In [13]:
def check_record_rolling(n):
    MOD = 10**9 + 7
    p0, l01, l02, p1, l11, l12 = 1, 0, 0, 0, 0, 0     # chuỗi rỗng
    for _ in range(n):
        s0 = (p0 + l01 + l02) % MOD                  # mọi chuỗi chưa vắng
        s1 = (p1 + l11 + l12) % MOD                  # mọi chuỗi đã vắng 1 lần
        p0, l01, l02, p1, l11, l12 = (
            s0,                 # thêm P vào chuỗi chưa vắng
            p0,                 # thêm L vào (0,0)
            l01,                # thêm L vào (0,1)
            (s0 + s1) % MOD,    # thêm A vào chuỗi chưa vắng + thêm P vào chuỗi đã vắng
            p1,                 # thêm L vào (1,0)
            l11,                # thêm L vào (1,1)
        )
    return (p0 + l01 + l02 + p1 + l11 + l12) % MOD

print([check_record_rolling(n) for n in range(1, 8)])
assert check_record_rolling(10101) == 183236316

[3, 8, 19, 43, 94, 200, 418]


**Đọc code:**

- `s0`, `s1` tính **trước** khi gán, nên dùng giá trị cũ.
- `p1` mới = `s0` (thêm `A`) + `s1` (thêm `P`). Đây là dòng dễ sai nhất: đừng quên phần thêm `A`.
- Các trạng thái "có `L` ở cuối" chỉ đến từ **đúng một** trạng thái trước (cùng `a`, `l` nhỏ hơn 1).

**Code nộp LeetCode (Cách 3):**

In [14]:
class SolutionRolling:
    def checkRecord(self, n: int) -> int:
        MOD = 10**9 + 7
        p0, l01, l02, p1, l11, l12 = 1, 0, 0, 0, 0, 0
        for _ in range(n):
            s0 = (p0 + l01 + l02) % MOD
            s1 = (p1 + l11 + l12) % MOD
            p0, l01, l02, p1, l11, l12 = s0, p0, l01, (s0 + s1) % MOD, p1, l11
        return (p0 + l01 + l02 + p1 + l11 + l12) % MOD

assert SolutionRolling().checkRecord(2) == 8
assert SolutionRolling().checkRecord(10101) == 183236316
print("SolutionRolling OK")

SolutionRolling OK


**Độ phức tạp Cách 3:** Time `O(n)`, Space `O(1)`.

### 7.3. Cách 4 — Tách chữ `A` ra (tổ hợp)

**Giải thích.** Một chuỗi hợp lệ có **0 hoặc 1** chữ `A`.

- **Không có `A`:** chuỗi chỉ gồm `P`, `L`, không chứa `LLL`. Gọi số chuỗi như vậy độ dài `i` là `f[i]`.
- **Có đúng 1 `A` ở vị trí `i`** (0-indexed): chữ `A` **cắt đứt** mọi chuỗi `L` (vì `A` không phải `L`). Nên phần trái (độ dài `i`) và phần phải (độ dài `n-1-i`) là **hai chuỗi P/L độc lập**, mỗi phần không chứa `LLL`. Số cách = `f[i] · f[n-1-i]`.

```text
Đáp án = f[n] + Σ_{i=0}^{n-1} f[i] · f[n-1-i]
```

**Tính `f`:** chuỗi P/L không có `LLL` kết thúc bằng `P`, `PL` hoặc `PLL` (hoặc là cả chuỗi `""`, `"L"`, `"LL"` ở đầu). Bỏ phần đuôi đó đi:

```text
f[0] = 1, f[1] = 2, f[2] = 4
f[i] = f[i-1] + f[i-2] + f[i-3]     (i >= 3)
```

Tính tay `n = 2`: `f = [1, 2, 4]`. Không `A`: `f[2] = 4`. Có `A` ở vị trí 0: `f[0]·f[1] = 2`; ở vị trí 1: `f[1]·f[0] = 2`. Tổng `4 + 2 + 2 = 8` ✓.

In [15]:
def count_PL(n):
    """f[i] = số chuỗi P/L độ dài i không chứa 'LLL', i = 0..n"""
    MOD = 10**9 + 7
    f = [1, 2, 4] + [0] * max(0, n - 2)
    for i in range(3, n + 1):
        f[i] = (f[i - 1] + f[i - 2] + f[i - 3]) % MOD
    return f[:n + 1]

# kiểm tra f bằng brute force
for i in range(0, 8):
    brute = sum(1 for t in product('PL', repeat=i) if 'LLL' not in ''.join(t))
    assert count_PL(i)[i] == brute, i
print("f:", count_PL(10))

def check_record_split(n):
    MOD = 10**9 + 7
    f = count_PL(n)
    ans = f[n]
    for i in range(n):
        ans = (ans + f[i] * f[n - 1 - i]) % MOD
    return ans

print([check_record_split(n) for n in range(1, 8)])
assert check_record_split(10101) == 183236316

f: [1, 2, 4, 7, 13, 24, 44, 81, 149, 274, 504]
[3, 8, 19, 43, 94, 200, 418]


**Đọc code:**

- `[1, 2, 4] + [0] * max(0, n - 2)` tạo mảng đủ dài kể cả khi `n < 2`; sau đó cắt `f[:n+1]`.
- Tích `f[i] * f[n-1-i]` có thể tới `~10^18` trước khi `% MOD` — Python xử lý thoải mái.
- Tổng có tính đối xứng (`i` và `n-1-i` cho cùng tích) nhưng không cần khai thác.

**Code nộp LeetCode (Cách 4):**

In [16]:
class SolutionSplit:
    def checkRecord(self, n: int) -> int:
        MOD = 10**9 + 7
        f = [1, 2, 4] + [0] * max(0, n - 2)
        for i in range(3, n + 1):
            f[i] = (f[i - 1] + f[i - 2] + f[i - 3]) % MOD
        ans = f[n]
        for i in range(n):
            ans = (ans + f[i] * f[n - 1 - i]) % MOD
        return ans

assert SolutionSplit().checkRecord(1) == 3
assert SolutionSplit().checkRecord(10101) == 183236316
print("SolutionSplit OK")

SolutionSplit OK


**Độ phức tạp Cách 4:** Time `O(n)`, Space `O(n)` (mảng `f`).

### 7.4. Cách 5 — Lũy thừa ma trận `O(log n)`

**Giải thích.** Cách 3 là một **phép biến đổi tuyến tính** trên vector 6 trạng thái: `v_{i+1} = M · v_i`. Vậy `v_n = M^n · v_0`, với `v_0 = (1,0,0,0,0,0)`. Lũy thừa nhanh ma trận 6×6 cần `O(6^3 · log n)` phép tính.

Hàng `r` của `M` ghi "trạng thái mới `r` lấy từ những trạng thái cũ nào". Thứ tự trạng thái: `[(0,0), (0,1), (0,2), (1,0), (1,1), (1,2)]`.

```text
          (0,0) (0,1) (0,2) (1,0) (1,1) (1,2)
(0,0) <- [  1     1     1     0     0     0  ]   thêm P
(0,1) <- [  1     0     0     0     0     0  ]   thêm L
(0,2) <- [  0     1     0     0     0     0  ]   thêm L
(1,0) <- [  1     1     1     1     1     1  ]   thêm A (từ a=0) hoặc P (từ a=1)
(1,1) <- [  0     0     0     1     0     0  ]
(1,2) <- [  0     0     0     0     1     0  ]
```

In [17]:
def build_matrix():
    states = [(a, l) for a in range(2) for l in range(3)]
    idx = {s: i for i, s in enumerate(states)}
    M = [[0] * 6 for _ in range(6)]
    for (a, l) in states:                     # cột = trạng thái cũ
        j = idx[(a, l)]
        M[idx[(a, 0)]][j] += 1                # P
        if a == 0:
            M[idx[(1, 0)]][j] += 1            # A
        if l < 2:
            M[idx[(a, l + 1)]][j] += 1        # L
    return M

M = build_matrix()
for row in M:
    print(row)

def check_record_matrix(n):
    MOD = 10**9 + 7
    P = mat_pow(build_matrix(), n, MOD)
    # v_n = P · v_0, v_0 = e_0  =>  v_n = cột 0 của P
    return sum(P[r][0] for r in range(6)) % MOD

print([check_record_matrix(n) for n in range(1, 8)])
assert check_record_matrix(10101) == 183236316
print("n = 10^18:", check_record_matrix(10**18))

[1, 1, 1, 0, 0, 0]
[1, 0, 0, 0, 0, 0]
[0, 1, 0, 0, 0, 0]
[1, 1, 1, 1, 1, 1]
[0, 0, 0, 1, 0, 0]
[0, 0, 0, 0, 1, 0]
[3, 8, 19, 43, 94, 200, 418]
n = 10^18: 665873632


**Đọc code:**

- Ta xây `M` **tự động** từ bảng chuyển — tránh gõ tay sai số.
- `v_0 = e_0` (vector đơn vị tại trạng thái `(0,0)`), nên `M^n · v_0` chính là **cột 0** của `M^n`.
- `mat_pow` từ mục 3.4: `O(log n)` phép nhân ma trận 6×6.

**Code nộp LeetCode (Cách 5):**

In [18]:
class SolutionMatrix:
    def checkRecord(self, n: int) -> int:
        MOD = 10**9 + 7
        M = [[1, 1, 1, 0, 0, 0],
             [1, 0, 0, 0, 0, 0],
             [0, 1, 0, 0, 0, 0],
             [1, 1, 1, 1, 1, 1],
             [0, 0, 0, 1, 0, 0],
             [0, 0, 0, 0, 1, 0]]

        def mul(A, B):
            return [[sum(A[i][k] * B[k][j] for k in range(6)) % MOD for j in range(6)] for i in range(6)]

        R = [[int(i == j) for j in range(6)] for i in range(6)]
        while n:
            if n & 1:
                R = mul(R, M)
            M = mul(M, M)
            n >>= 1
        return sum(R[i][0] for i in range(6)) % MOD

assert SolutionMatrix().checkRecord(2) == 8
assert SolutionMatrix().checkRecord(10101) == 183236316
assert build_matrix() == [[1, 1, 1, 0, 0, 0], [1, 0, 0, 0, 0, 0], [0, 1, 0, 0, 0, 0],
                          [1, 1, 1, 1, 1, 1], [0, 0, 0, 1, 0, 0], [0, 0, 0, 0, 1, 0]]
print("SolutionMatrix OK")

SolutionMatrix OK


**Độ phức tạp Cách 5:** Time `O(6^3 log n) = O(log n)`, Space `O(1)` (ma trận cố định 6×6).

## 8. TRACE từng bước

Trace **Cách 3** với `n = 4`. Bảng trạng thái sau mỗi bước (tính tay theo công thức `s0 = p0+l01+l02`, `s1 = p1+l11+l12`):

| bước | p0 | l01 | l02 | p1 | l11 | l12 | tổng |
|---|---|---|---|---|---|---|---|
| 0 | 1 | 0 | 0 | 0 | 0 | 0 | 1 |
| 1 | 1 | 1 | 0 | 1 | 0 | 0 | 3 |
| 2 | 2 | 1 | 1 | 3 | 1 | 0 | 8 |
| 3 | 4 | 2 | 1 | 8 | 3 | 1 | 19 |
| 4 | 7 | 4 | 2 | 19 | 8 | 3 | 43 |

Ví dụ bước 3: `s0 = 2+1+1 = 4`, `s1 = 3+1+0 = 4` → `p0 = 4`, `l01 = 2`, `l02 = 1`, `p1 = 4+4 = 8`, `l11 = 3`, `l12 = 1`.

In [19]:
def trace_rolling(n):
    p0, l01, l02, p1, l11, l12 = 1, 0, 0, 0, 0, 0
    print(f"{'bước':>4} | {'p0':>3} {'l01':>3} {'l02':>3} | {'p1':>3} {'l11':>3} {'l12':>3} | {'s0':>3} {'s1':>3} | tổng")
    print(f"{0:>4} | {p0:>3} {l01:>3} {l02:>3} | {p1:>3} {l11:>3} {l12:>3} | {'':>3} {'':>3} | {p0+l01+l02+p1+l11+l12}")
    for i in range(1, n + 1):
        s0 = p0 + l01 + l02
        s1 = p1 + l11 + l12
        p0, l01, l02, p1, l11, l12 = s0, p0, l01, s0 + s1, p1, l11
        total = p0 + l01 + l02 + p1 + l11 + l12
        print(f"{i:>4} | {p0:>3} {l01:>3} {l02:>3} | {p1:>3} {l11:>3} {l12:>3} | {s0:>3} {s1:>3} | {total}")
    return total

assert trace_rolling(4) == check_record_brute(4) == 43

bước |  p0 l01 l02 |  p1 l11 l12 |  s0  s1 | tổng
   0 |   1   0   0 |   0   0   0 |         | 1
   1 |   1   1   0 |   1   0   0 |   1   0 | 3
   2 |   2   1   1 |   3   1   0 |   2   1 | 8
   3 |   4   2   1 |   8   3   1 |   4   4 | 19
   4 |   7   4   2 |  19   8   3 |   7  12 | 43


Trace thêm: liệt kê chuỗi thật theo trạng thái cho `n = 3` để thấy mỗi con số trong bảng là **bao nhiêu chuỗi cụ thể**:

In [20]:
def trace_strings(n):
    groups = {}
    for t in product('PAL', repeat=n):
        s = ''.join(t)
        if is_rewardable(s):
            groups.setdefault(state_of(s), []).append(s)
    for st in sorted(groups):
        print(st, len(groups[st]), groups[st])

trace_strings(3)

(0, 0) 4 ['PPP', 'PLP', 'LPP', 'LLP']
(0, 1) 2 ['PPL', 'LPL']
(0, 2) 1 ['PLL']
(1, 0) 8 ['PPA', 'PAP', 'PLA', 'APP', 'ALP', 'LPA', 'LAP', 'LLA']
(1, 1) 3 ['PAL', 'APL', 'LAL']
(1, 2) 1 ['ALL']


Khớp hàng "bước 3": `(0,0)`: 4, `(0,1)`: 2, `(0,2)`: 1, `(1,0)`: 8, `(1,1)`: 3, `(1,2)`: 1.

## 9. Code nộp LeetCode

Chọn Cách 3 (6 biến): ngắn, `O(n)` thời gian, `O(1)` bộ nhớ, đủ nhanh cho `n = 10^5`.

In [21]:
class Solution:
    def checkRecord(self, n: int) -> int:
        MOD = 10**9 + 7
        # (a, l): a = số 'A' đã dùng, l = số 'L' liên tiếp ở cuối
        p0, l01, l02 = 1, 0, 0      # a = 0, l = 0/1/2
        p1, l11, l12 = 0, 0, 0      # a = 1, l = 0/1/2
        for _ in range(n):
            s0 = (p0 + l01 + l02) % MOD
            s1 = (p1 + l11 + l12) % MOD
            p0, l01, l02, p1, l11, l12 = s0, p0, l01, (s0 + s1) % MOD, p1, l11
        return (p0 + l01 + l02 + p1 + l11 + l12) % MOD

sol = Solution()
assert sol.checkRecord(1) == 3
assert sol.checkRecord(2) == 8
assert sol.checkRecord(10101) == 183236316
t = time.perf_counter()
print("n=10^5:", sol.checkRecord(10**5), f"({time.perf_counter() - t:.3f}s)")

n=10^5: 749184020 (0.016s)


## 10. Vì sao thuật toán đúng

**Bổ đề (đóng gói trạng thái).** Một chuỗi `s` hợp lệ khi và chỉ khi mọi tiền tố của nó hợp lệ (vì hai điều kiện "ít hơn 2 chữ A" và "không có LLL" một khi vi phạm thì không bao giờ phục hồi). Khi nối thêm ký tự `c` vào tiền tố hợp lệ `t`, chuỗi `t + c` hợp lệ **chỉ phụ thuộc** `(a(t), l(t))` và `c`:
- `c = A`: hợp lệ ⇔ `a(t) = 0`.
- `c = L`: hợp lệ ⇔ `l(t) < 2` (chuỗi L ở cuối dài thêm 1).
- `c = P`: luôn hợp lệ.

Và trạng thái mới `(a(t+c), l(t+c))` cũng chỉ phụ thuộc `(a(t), l(t))` và `c`.

**Quy nạp.** Giả sử sau `i` bước, `dp[a][l]` đúng bằng số chuỗi độ dài `i` hợp lệ ở trạng thái `(a, l)`. Mỗi chuỗi hợp lệ độ dài `i+1` có **duy nhất** một cách viết `t + c` với `t` hợp lệ độ dài `i`. Công thức chuyển cộng đúng một lần cho mỗi cặp `(t, c)` hợp lệ → `dp` sau bước `i+1` cũng đúng. Bước 0: chỉ có chuỗi rỗng ở `(0,0)`.

**Cách 4 đúng** vì: chuỗi có 0 chữ A ⇔ chuỗi P/L không `LLL` (`f[n]`); chuỗi có đúng 1 chữ A ở vị trí `i` ⇔ (trái: P/L không `LLL`) + `A` + (phải: P/L không `LLL`), vì `A` tách hai bên nên không chuỗi `LLL` nào vắt qua nó. Các vị trí `i` khác nhau cho các tập chuỗi rời nhau → cộng lại.

**Cách 5 đúng** vì phép cập nhật là tuyến tính, nên `n` lần áp dụng = nhân với `M^n`.

## 11. Phân tích Time/Space

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Brute force | `O(3^n · n)` | `O(n)` | chỉ `n <= 12` |
| 2. Bảng `dp[i][a][l]` | `O(n)` | `O(n)` | 6 ô mỗi tầng |
| 3. 6 biến | `O(n)` | `O(1)` | **nộp bài** |
| 4. Tách chữ A | `O(n)` | `O(n)` | đẹp về tổ hợp |
| 5. Lũy thừa ma trận | `O(216 · log n)` | `O(1)` | khi `n` cực lớn |

Đo thời gian thực tế với `n = 10^5`:

In [22]:
for name, f in [("table", check_record_table), ("rolling", check_record_rolling),
                ("split", check_record_split), ("matrix", check_record_matrix)]:
    t = time.perf_counter()
    r = f(10**5)
    print(f"{name:>8}: {r}  {time.perf_counter() - t:.3f}s")

   table: 749184020  0.224s
 rolling: 749184020  0.016s
   split: 749184020  0.018s
  matrix: 749184020  0.000s


## 12. Bộ test

In [23]:
impls = [check_record_table, check_record_rolling, check_record_split, check_record_matrix,
         lambda n: SolutionTable().checkRecord(n), lambda n: SolutionRolling().checkRecord(n),
         lambda n: SolutionSplit().checkRecord(n), lambda n: SolutionMatrix().checkRecord(n),
         lambda n: Solution().checkRecord(n)]

# Ví dụ đề
for f in impls:
    assert f(1) == 3 and f(2) == 8 and f(10101) == 183236316
print("Ví dụ đề OK")

# Đối chiếu với brute force cho n nhỏ (edge case n = 1, 2, 3 nằm trong đây)
for n in range(1, 11):
    b = check_record_brute(n)
    for f in impls:
        assert f(n) == b, (n, f)
print("Khớp brute force với n = 1..10")

# Ngẫu nhiên có seed: các cách nhanh khớp nhau với n lớn
import random
random.seed(552)
for _ in range(15):
    n = random.randint(11, 3000)
    res = {f(n) for f in impls}
    assert len(res) == 1, n
print("15 test ngẫu nhiên n = 11..3000: khớp")

# Biên lớn nhất
assert len({f(10**5) for f in [check_record_rolling, check_record_split, check_record_matrix]}) == 1
print("n = 10^5 khớp giữa 3 cách")

Ví dụ đề OK
Khớp brute force với n = 1..10


15 test ngẫu nhiên n = 11..3000: khớp
n = 10^5 khớp giữa 3 cách


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Cập nhật tuần tự, vế phải dùng giá trị mới

In [24]:
def wrong_sequential(n):
    p0, l01, l02, p1, l11, l12 = 1, 0, 0, 0, 0, 0
    for _ in range(n):
        p0 = p0 + l01 + l02          # p0 đã bị ghi đè...
        l01 = p0                     # ...nên l01 lấy p0 MỚI (sai)
        l02 = l01
        p1 = p0 + p1 + l11 + l12
        l11 = p1
        l12 = l11
    return p0 + l01 + l02 + p1 + l11 + l12

print("sai:", wrong_sequential(2), "| đúng:", Solution().checkRecord(2))

sai: 27 | đúng: 8


**Vì sao sai:** mỗi dòng dùng giá trị vừa cập nhật ở dòng trên, nên một chuỗi bị đếm như thể được thêm nhiều ký tự trong cùng một bước.

**Code đúng:** tính `s0`, `s1` trước, rồi **gán đồng thời** (`Solution` ở mục 9), hoặc tạo biến mới `n_p0, n_l01, ...` rồi mới gán.

### Lỗi 2 — Hiểu sai "3 L liên tiếp" thành "tổng số L ít hơn 3"

In [25]:
def wrong_total_L(n):
    return sum(1 for t in product('PAL', repeat=n)
               if t.count('A') < 2 and t.count('L') < 3)

print("n=4 sai:", wrong_total_L(4), "| đúng:", check_record_brute(4))
print("'LLPLL' hợp lệ thật không?", is_rewardable("LLPLL"))

n=4 sai: 39 | đúng: 43
'LLPLL' hợp lệ thật không? True


**Vì sao sai:** đề chỉ cấm **3 ngày muộn liên tiếp**. `"LLPLL"` có 4 chữ L nhưng vẫn hợp lệ. Trạng thái phải là "số L **liên tiếp ở cuối**", và **reset về 0** khi gặp `P` hoặc `A`.

**Code đúng:** trạng thái `l` reset khi thêm `P`/`A`, như mục 4.2.

### Lỗi 3 — Quên rằng thêm `A` cũng reset chuỗi `L`

In [26]:
def wrong_A_keeps_L(n):
    from collections import Counter
    cnt = Counter({(0, 0): 1})
    for _ in range(n):
        new = Counter()
        for (a, l), c in cnt.items():
            new[(a, 0)] += c
            if a == 0:
                new[(1, l)] += c        # SAI: giữ nguyên l khi thêm A
            if l < 2:
                new[(a, l + 1)] += c
        cnt = new
    return sum(cnt.values())

print("n=4 sai:", wrong_A_keeps_L(4), "| đúng:", check_record_brute(4))

n=4 sai: 41 | đúng: 43


**Vì sao sai:** `"LLAL"` là hợp lệ (chữ `A` cắt chuỗi `L`), nhưng code sai tưởng sau `A` vẫn còn 2 L ở cuối nên cấm thêm `L` → đếm thiếu.

**Code đúng:** thêm `A` → trạng thái `(1, 0)`.

### Lỗi 4 — Chỉ lấy modulo ở cuối (hoặc quên hẳn)

In [27]:
def no_mod(n):
    p0, l01, l02, p1, l11, l12 = 1, 0, 0, 0, 0, 0
    for _ in range(n):
        s0, s1 = p0 + l01 + l02, p1 + l11 + l12
        p0, l01, l02, p1, l11, l12 = s0, p0, l01, s0 + s1, p1, l11
    return (p0 + l01 + l02 + p1 + l11 + l12) % (10**9 + 7)

for n in (2000, 8000):
    t = time.perf_counter(); r1 = no_mod(n); t1 = time.perf_counter() - t
    t = time.perf_counter(); r2 = Solution().checkRecord(n); t2 = time.perf_counter() - t
    assert r1 == r2
    print(f"n={n}: không mod {t1:.4f}s | mod mỗi bước {t2:.4f}s")

n=2000: không mod 0.0004s | mod mỗi bước 0.0003s
n=8000: không mod 0.0048s | mod mỗi bước 0.0016s


**Vì sao không nên:** trong Python kết quả vẫn đúng (số nguyên không tràn) nhưng các số có **hàng nghìn chữ số** khiến mỗi phép cộng chậm dần — `n = 10^5` có thể bị TLE. Trong C++/Java thì **tràn số** và sai luôn.

**Code đúng:** `% MOD` sau mỗi bước.

## 14. Mẹo & nhận dạng pattern

- **"Đếm số chuỗi độ dài `n` thỏa điều kiện cục bộ"** (giới hạn số lần xuất hiện, cấm `k` ký tự liên tiếp, cấm một mẫu con) → **DP máy trạng thái**: trạng thái = thông tin tối thiểu về tiền tố quyết định tương lai.
- **Đếm trạng thái:** "tối đa `x` lần" → `x+1` giá trị; "không quá `k` liên tiếp" → `k` giá trị (0..k-1). Tổng trạng thái = tích.
- **Kết quả modulo `10^9+7`** gần như luôn là tín hiệu "đây là bài đếm bằng DP".
- **Phần tử "phân cách"** (như chữ `A` cắt chuỗi `L`) → có thể tách bài toán: đếm phần không có nó, rồi nhân trái × phải.
- **Chuyển trạng thái tuyến tính, hệ số cố định** + `n` khổng lồ → **lũy thừa ma trận**.

## 15. Biến thể

1. **LeetCode 551 — Student Attendance Record I:** chỉ kiểm tra **một** chuỗi có hợp lệ không (`is_rewardable`).
2. **Cho phép tối đa `maxA` lần vắng, cấm `maxL` ngày muộn liên tiếp:** trạng thái `(a, l)` với `a < maxA`, `l < maxL` → `maxA · maxL` trạng thái.
3. **Đếm chuỗi có đúng một chữ A:** dùng lại Cách 4, bỏ phần `f[n]`.
4. **Count Vowels Permutation (LeetCode 1220), Knight Dialer (LeetCode 935):** cùng mô hình "máy trạng thái + đếm".
5. **`n` tới `10^18`:** bắt buộc Cách 5.

In [28]:
# Biến thể 2: tổng quát hóa maxA, maxL
def check_record_general(n, maxA=2, maxL=3):
    """Đếm chuỗi có < maxA chữ 'A' và không có maxL chữ 'L' liên tiếp."""
    MOD = 10**9 + 7
    dp = [[0] * maxL for _ in range(maxA)]
    dp[0][0] = 1
    for _ in range(n):
        new = [[0] * maxL for _ in range(maxA)]
        for a in range(maxA):
            for l in range(maxL):
                c = dp[a][l]
                if not c:
                    continue
                new[a][0] = (new[a][0] + c) % MOD               # P
                if a + 1 < maxA:
                    new[a + 1][0] = (new[a + 1][0] + c) % MOD   # A
                if l + 1 < maxL:
                    new[a][l + 1] = (new[a][l + 1] + c) % MOD   # L
        dp = new
    return sum(map(sum, dp)) % MOD

assert check_record_general(10101) == 183236316
for n in range(1, 7):
    brute = sum(1 for t in product('PAL', repeat=n) if t.count('A') < 3 and 'LLLL' not in ''.join(t))
    assert check_record_general(n, 3, 4) == brute
print("Tổng quát hóa OK; maxA=3, maxL=4, n=5:", check_record_general(5, 3, 4))

Tổng quát hóa OK; maxA=3, maxL=4, n=5: 187


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | Sinh `3^n` chuỗi, kiểm tra | `O(3^n · n)` | `O(n)` | Kiểm chứng `n` nhỏ |
| 2. Bảng `dp[i][a][l]` | Máy trạng thái, lưu mọi tầng | `O(n)` | `O(n)` | Học / debug từng tầng |
| 3. 6 biến | Như 2, chỉ giữ tầng hiện tại | `O(n)` | `O(1)` | **Nộp bài** |
| 4. Tách chữ A | `f[n] + Σ f[i]·f[n-1-i]` | `O(n)` | `O(n)` | Khi thấy phần tử phân cách |
| 5. Lũy thừa ma trận | `v_n = M^n v_0` | `O(log n)` | `O(1)` | `n` cực lớn |

## 17. Bài tập

**Bài 1 (đoán output).** Không chạy code: `Solution().checkRecord(3)` bằng bao nhiêu? (Gợi ý: dùng bảng trace mục 8.)

**Bài 2 (tính tay).** Tính tay `f[0..6]` (số chuỗi P/L không `LLL`), rồi dùng Cách 4 tính đáp án cho `n = 3`.

**Bài 3 (điền TODO).** Viết `is_rewardable_scan(s)` kiểm tra một chuỗi **bằng một lần duyệt**, không dùng `count` hay `in` (chính là bài 551).

**Bài 4 (tự viết lại).** Viết `count_exactly_one_A(n)`: số chuỗi hợp lệ có **đúng một** chữ `A`, dùng DP 6 biến (không dùng Cách 4).

**Bài 5 (mini challenge).** Viết `kth_record(n, k)`: trả về chuỗi hợp lệ thứ `k` (đánh số từ 0) theo thứ tự từ điển `A < L < P`. Gợi ý: cần bảng "số cách viết tiếp `m` ký tự từ trạng thái `(a, l)`".

In [29]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def is_rewardable_scan(s):
    absent = 0
    late_run = 0
    for ch in s:
        # TODO: cập nhật absent, late_run; trả False ngay khi vi phạm
        pass
    return True

# Bài 4 — TODO
def count_exactly_one_A(n):
    # TODO: chạy DP 6 biến như Solution, nhưng trả về tổng 3 trạng thái a = 1
    pass

# Bài 5 — TODO
def kth_record(n, k):
    # TODO: ways[m][a][l] = số cách viết thêm m ký tự từ trạng thái (a, l)
    #       rồi chọn từng ký tự theo thứ tự 'A', 'L', 'P'
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: xem hàng bước 3 của bảng.
- Mức 2: cộng 6 số `4, 2, 1, 8, 3, 1`.
- Mức 3: 19.

**Bài 2.**
- Mức 1: `f[0..2] = 1, 2, 4`.
- Mức 2: `f[3] = 7`, tiếp tục cộng 3 số trước.
- Mức 3: `n = 3`: `f[3] + f[0]f[2] + f[1]f[1] + f[2]f[0] = 7 + 4 + 4 + 4`.

**Bài 3.**
- Mức 1: gặp `A` → `absent += 1`.
- Mức 2: gặp `L` → `late_run += 1`, ngược lại `late_run = 0`.
- Mức 3: vi phạm khi `absent >= 2` hoặc `late_run >= 3`.

**Bài 4.**
- Mức 1: cuối vòng lặp, `p1 + l11 + l12` là số chuỗi có đúng 1 chữ A.
- Mức 2: không cần đổi gì trong vòng lặp.
- Mức 3: đối chiếu với Cách 4 bỏ `f[n]`.

**Bài 5.**
- Mức 1: `ways[0][a][l] = 1`; `ways[m][a][l] = ways[m-1][a][0] (P) + [a==0]·ways[m-1][1][0] (A) + [l<2]·ways[m-1][a][l+1] (L)`.
- Mức 2: ở mỗi vị trí, thử ký tự theo thứ tự `A, L, P`; nếu `k < số cách` với ký tự đó thì chọn, ngược lại trừ `k` đi.
- Mức 3: không lấy modulo (cần số chính xác); chỉ dùng với `n` vừa phải.

## 19. Đáp án tham khảo

**Bài 1:** 19. **Bài 2:** `f = [1, 2, 4, 7, 13, 24, 44]`; `n = 3`: `7 + 4 + 4 + 4 = 19`.

In [30]:
assert Solution().checkRecord(3) == 19
assert count_PL(6) == [1, 2, 4, 7, 13, 24, 44]
assert check_record_split(3) == 7 + 4 + 4 + 4

# Bài 3
def is_rewardable_scan(s):
    absent = late_run = 0
    for ch in s:
        if ch == 'A':
            absent += 1
            if absent >= 2:
                return False
        if ch == 'L':
            late_run += 1
            if late_run >= 3:
                return False
        else:
            late_run = 0
    return True

for n in range(0, 7):
    for t in product('PAL', repeat=n):
        s = ''.join(t)
        assert is_rewardable_scan(s) == is_rewardable(s)
print("Bài 3 OK")

Bài 3 OK


In [31]:
# Bài 4
def count_exactly_one_A(n):
    MOD = 10**9 + 7
    p0, l01, l02, p1, l11, l12 = 1, 0, 0, 0, 0, 0
    for _ in range(n):
        s0 = (p0 + l01 + l02) % MOD
        s1 = (p1 + l11 + l12) % MOD
        p0, l01, l02, p1, l11, l12 = s0, p0, l01, (s0 + s1) % MOD, p1, l11
    return (p1 + l11 + l12) % MOD

for n in range(1, 9):
    brute = sum(1 for t in product('PAL', repeat=n) if t.count('A') == 1 and 'LLL' not in ''.join(t))
    assert count_exactly_one_A(n) == brute
    assert count_exactly_one_A(n) == (check_record_split(n) - count_PL(n)[n]) % MOD
print("Bài 4 OK")

Bài 4 OK


In [32]:
# Bài 5
def kth_record(n, k):
    ways = [[[1] * 3 for _ in range(2)]]           # m = 0
    for m in range(1, n + 1):
        prev = ways[-1]
        cur = [[0] * 3 for _ in range(2)]
        for a in range(2):
            for l in range(3):
                cur[a][l] = prev[a][0] + (prev[1][0] if a == 0 else 0) + (prev[a][l + 1] if l < 2 else 0)
        ways.append(cur)
    if k >= ways[n][0][0]:
        return None
    a = l = 0
    res = []
    for pos in range(n):
        rem = n - pos - 1
        for ch in 'ALP':
            if ch == 'A':
                if a == 1:
                    continue
                na, nl = 1, 0
            elif ch == 'L':
                if l == 2:
                    continue
                na, nl = a, l + 1
            else:
                na, nl = a, 0
            c = ways[rem][na][nl]
            if k < c:
                res.append(ch)
                a, l = na, nl
                break
            k -= c
    return ''.join(res)

for n in range(1, 7):
    all_good = sorted(''.join(t) for t in product('PAL', repeat=n) if is_rewardable(''.join(t)))
    assert [kth_record(n, k) for k in range(len(all_good))] == all_good
    assert kth_record(n, len(all_good)) is None
print("Bài 5 OK; n=5, k=10:", kth_record(5, 10))

Bài 5 OK; n=5, k=10: APPLP


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "DP máy trạng thái đếm chuỗi":**

```python
MOD = 10**9 + 7
dp = {start_state: 1}
for _ in range(n):
    new = defaultdict(int)
    for state, cnt in dp.items():
        for ch in alphabet:
            if allowed(state, ch):
                ns = transition(state, ch)
                new[ns] = (new[ns] + cnt) % MOD
    dp = new
answer = sum(dp.values()) % MOD
```

Khi đã rõ các trạng thái → viết tay thành vài biến và **gán đồng thời**.

**Câu thần chú:** *"Nhớ đủ để quyết định tương lai, không nhớ hơn."*

**Checklist trước khi submit:**

- [ ] Liệt kê đủ trạng thái: `a ∈ {0,1}`, `l ∈ {0,1,2}` → 6.
- [ ] `P` và `A` đều **reset** `l` về 0.
- [ ] `A` chỉ thêm được khi `a = 0`; `L` chỉ thêm được khi `l < 2`.
- [ ] Cập nhật bằng **gán đồng thời** (vế phải là giá trị cũ).
- [ ] `% MOD` sau mỗi bước.
- [ ] Thử `n = 1 → 3`, `n = 2 → 8`, `n = 3 → 19`, `n = 10101 → 183236316`.
- [ ] Đối chiếu brute force với `n <= 10`.